# Orth Risk walkthrough — stocks, funds, and hedge-depth dispatch

**[Get API key](https://riskmodels.app/get-key)** · **[Open in Colab](https://colab.research.google.com/github/BlueWaterCorp/RiskModels_API/blob/main/sdk/notebooks/orth_risk_walkthrough.ipynb)** · `POST /decompose` · `GET /lstar` · `GET /funds/*`

ERM3 estimates one hierarchical sequence, bottom-up, on ~3,100 US names daily:

**market → sector → subsector → FF2 style → final residual**

Each layer is the incremental strip after the one above, so exposures never overlap.
**Tradeable hedge legs stop at subsector** — market, sector, and subsector each map to a
single liquid ETF with a signed hedge ratio in dollars. **The FF2 block (SMB + HML) carries
no ETF hedge** — it is an attribution layer that keeps style tilt out of the stock-selection
read.

| ERM3 Orth Risk | Cross-sectional factor models |
|---|---|
| Hierarchical orthogonalization — each layer is the incremental regression after the one above | Jointly estimated factors; exposures overlap |
| Market / sector / subsector each hedgeable with **one ETF**; signed HR × notional = a ticket | Factor-mimicking portfolios; not directly tradeable |
| Explained Risk is a **signed covariance share** — layers sum to 100%, and a single layer can print negative | R²-style attribution hides sign |
| **L\*** dispatches the shallowest hedge depth whose marginal ER clears a threshold | Depth choice left to the user |
| **Final residual** — the selection sleeve after the industry cascade *and* the FF2 strip | Style-box heuristics |

This notebook walks the whole hierarchy end to end: a stock's return attribution and
signed risk, why a layer can go negative, the L\* hedge-depth call, a mutual fund's
holdings-derived cascade (through the FF2 layer), and a small "bring your book" portfolio.

For a deep dive on any one piece, jump to a sibling notebook — links at the bottom.


### Colab only (skip locally)


In [ ]:
import sys

try:
    import google.colab  # noqa: F401

    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    import subprocess

    _deps = ["requests", "python-dotenv"]
    _pypi = "riskmodels-py>=0.3.6,<0.4"
    _git = (
        "riskmodels-py @ git+https://github.com/BlueWaterCorp/RiskModels_API.git"
        "@main#subdirectory=sdk"
    )
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _pypi, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py from PyPI (+ requests, python-dotenv).")
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _git, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: PyPI wheel not available yet — installed riskmodels-py from GitHub main.")


## Connect

`quickstart_connect()` looks for `RISKMODELS_API_KEY` in the environment, a local `.env` / `.env.local`, or Colab Secrets. If missing, it prompts securely; the key is never printed.


In [ ]:
import os
from pathlib import Path
from typing import Any

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from IPython.display import Image, display

from riskmodels import RiskModelsClient
from riskmodels.notebook import quickstart_connect

session, BASE_URL, API_KEY = quickstart_connect()
os.environ.setdefault("RISKMODELS_API_KEY", API_KEY)
os.environ.setdefault("RISKMODELS_BASE_URL", BASE_URL)
client = RiskModelsClient.from_env()

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Shared visual language ────────────────────────────────────────────────
NAVY = "#1B2A4A"
LAYER_COLORS = {
    "Market": "#64748B",
    "Sector": "#1D6FA8",
    "Subsector": "#7C3AED",
    "Style (FF2)": "#0D9488",     # attribution only — no ETF hedge
    "Residual": "#16A34A",
    "Final residual": "#16A34A",
    "Gross": NAVY,
}
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlecolor": NAVY, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#E5E9F0", "grid.linewidth": 0.8,
    "axes.edgecolor": "#94A3B8", "axes.labelcolor": "#334155",
    "xtick.color": "#475569", "ytick.color": "#475569",
    "legend.frameon": False,
})


def api_get(path: str, **params: Any) -> tuple[dict, dict]:
    response = session.get(f"{BASE_URL}{path}", params=params, timeout=90)
    if not response.ok:
        raise RuntimeError(f"GET {path} failed ({response.status_code}): {response.text[:500]}")
    headers = {
        "data_as_of": response.headers.get("X-Data-As-Of"),
        "filing_date": response.headers.get("X-Data-Filing-Date"),
        "cost_usd": response.headers.get("X-API-Cost-USD"),
    }
    return response.json(), headers


def api_post(path: str, **body: Any) -> tuple[dict, dict]:
    response = session.post(f"{BASE_URL}{path}", json=body, timeout=90)
    if not response.ok:
        raise RuntimeError(f"POST {path} failed ({response.status_code}): {response.text[:500]}")
    return response.json(), {"cost_usd": response.headers.get("X-API-Cost-USD")}


def pick_col(frame: pd.DataFrame, *names: str) -> str | None:
    return next((name for name in names if name in frame.columns), None)


def waterfall(ax, steps: dict, total_label="Gross", title=None):
    """Stacked waterfall: ordered {layer: contribution}. Bars float at the running
    cumulative, residual layers hatched, total bar in navy, connectors dotted."""
    cum = 0.0
    for i, (lab, v) in enumerate(steps.items()):
        hatch = "//" if "residual" in lab.lower() else None
        ax.bar(i, v, bottom=cum, width=0.62, color=LAYER_COLORS.get(lab, "#94A3B8"),
               hatch=hatch, edgecolor="white", linewidth=0.5)
        ax.annotate(f"{v:+.1%}", (i, cum + v), ha="center",
                    va="bottom" if v >= 0 else "top", fontsize=9, fontweight="bold",
                    color="#334155" if v >= 0 else "#B45309")
        cum += v
        ax.plot([i + 0.31, i + 0.69], [cum, cum], ls=":", color="#9CA3AF", lw=1)
    n = len(steps)
    ax.bar(n, cum, width=0.62, color=NAVY)
    ax.annotate(f"{cum:+.1%}", (n, max(cum, 0.0)), ha="center", va="bottom",
                fontsize=10, fontweight="bold", color=NAVY)
    ax.set_xticks(range(n + 1))
    ax.set_xticklabels(list(steps) + [total_label], fontsize=9)
    levels = np.concatenate([[0.0], np.cumsum(list(steps.values()))])
    hi, lo = float(levels.max()), float(min(0.0, levels.min()))
    span = (hi - lo) or 1.0
    ax.set_ylim(lo - (0.12 * span if lo < 0 else 0.0), hi + 0.15 * span)
    ax.axhline(0, color="#9CA3AF", lw=0.8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    if title:
        ax.set_title(title, loc="left")


def cum_with_t0(contrib: pd.DataFrame, dates) -> pd.DataFrame:
    """Arithmetic cumulative attribution anchored at 0: prepends a zero row one
    business day before the first date so every line starts at exactly 0."""
    dates = pd.to_datetime(pd.Series(list(dates)).reset_index(drop=True))
    cum = contrib.reset_index(drop=True).fillna(0).cumsum()
    cum.index = dates
    t0 = dates.iloc[0] - pd.tseries.offsets.BDay(1)
    return pd.concat([pd.DataFrame(0.0, index=[t0], columns=cum.columns), cum])


def attribution_lines(ax, frame: pd.DataFrame, title=None):
    """Cumulative attribution lines (frame indexed by date, anchored at 0).
    Gross heavy navy; layers in the shared palette; dodged end-value labels."""
    dates = frame.index
    span = float(np.nanmax(frame.values) - np.nanmin(frame.values)) or 1.0
    for col in frame.columns:
        lw, color = (2.4, NAVY) if col == "Gross" else (1.5, LAYER_COLORS.get(col, "#94A3B8"))
        ax.plot(dates, frame[col], color=color, lw=lw)
    y_last = None
    for col, v in frame.iloc[-1].sort_values().items():
        y = v if y_last is None else max(v, y_last + 0.055 * span)
        y_last = y
        color = NAVY if col == "Gross" else LAYER_COLORS.get(col, "#94A3B8")
        ax.annotate(f" {col} {v:+.1%}", (dates[-1], y), color=color,
                    fontsize=8.5, va="center", fontweight="bold")
    ax.axhline(0, color="#9CA3AF", lw=0.8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlim(dates[0], dates[-1] + (dates[-1] - dates[0]) * 0.24)
    last_num = mdates.date2num(dates[-1])
    ax.set_xticks([t for t in ax.get_xticks() if t <= last_num])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    if title:
        ax.set_title(title, loc="left")


print("Connected to", BASE_URL)


## Parameters

Override with env vars: `ORTH_STOCK`, `ORTH_YEARS`, `ORTH_FUND`, `ORTH_NOTIONAL`.


In [ ]:
STOCK_TICKER = os.environ.get("ORTH_STOCK", "NVDA").upper()
STOCK_YEARS = int(os.environ.get("ORTH_YEARS", "3"))
FUND_QUERY = os.environ.get("ORTH_FUND", "AGTHX")
NOTIONAL_USD = float(os.environ.get("ORTH_NOTIONAL", "10000000"))
BOOK = {"NVDA": 0.25, "AAPL": 0.25, "MSFT": 0.25, "JPM": 0.25}

print({"stock": STOCK_TICKER, "years": STOCK_YEARS, "fund": FUND_QUERY, "notional": NOTIONAL_USD})


## Part I · Stock — the whole model in one picture

One call (`get_ticker_returns`) applies the full hierarchy to a window of realized
returns. Left: arithmetic cumulative attribution (`cumsum`), every line anchored at 0.
Right: the same window totalled as a waterfall — lines and bars agree by construction.
*Residual here is the L3 industry residual, pre-FF2.*


In [ ]:
stock_history = client.get_ticker_returns(STOCK_TICKER, years=STOCK_YEARS).copy()
if stock_history.empty:
    raise ValueError(f"No return history returned for {STOCK_TICKER}.")

stock_history["date"] = pd.to_datetime(stock_history["date"])
stock_history = stock_history.sort_values("date").reset_index(drop=True)

gross_col = pick_col(stock_history, "returns_gross", "gross_return")
l1_fr = pick_col(stock_history, "l1_factor_return", "l1_fr")
l2_fr = pick_col(stock_history, "l2_factor_return", "l2_fr")
l3_fr = pick_col(stock_history, "l3_factor_return", "l3_fr")
l1_cfr = pick_col(stock_history, "l1_combined_factor_return", "l1_cfr")
l2_cfr = pick_col(stock_history, "l2_combined_factor_return", "l2_cfr")
l3_cfr = pick_col(stock_history, "l3_combined_factor_return", "l3_cfr")
l3_rr = pick_col(stock_history, "l3_residual_return", "l3_rr")

if gross_col is None or l3_cfr is None:
    raise KeyError("Expected gross and L3 factor-return fields are not present. Inspect stock_history.columns.")

contrib = pd.DataFrame(index=stock_history.index)
contrib["Market"] = stock_history[l1_fr] if l1_fr else stock_history[l1_cfr]
contrib["Sector"] = stock_history[l2_fr] if l2_fr else stock_history[l2_cfr] - stock_history[l1_cfr]
contrib["Subsector"] = stock_history[l3_fr] if l3_fr else stock_history[l3_cfr] - stock_history[l2_cfr]
contrib["Residual"] = stock_history[l3_rr] if l3_rr else stock_history[gross_col] - stock_history[l3_cfr]
contrib["Gross"] = stock_history[gross_col]

cumulative = cum_with_t0(contrib, stock_history["date"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [2.2, 1]})
attribution_lines(ax1, cumulative,
                  title=f"{STOCK_TICKER} — cumulative return attribution ({STOCK_YEARS}y, arithmetic)")
totals = contrib[["Market", "Sector", "Subsector", "Residual"]].sum()
waterfall(ax2, totals.to_dict(), total_label="Gross", title="Same window, totalled")
plt.tight_layout()
plt.show()

check = (contrib[["Market", "Sector", "Subsector", "Residual"]].sum(axis=1) - contrib["Gross"]).abs()
print("Mean absolute daily identity gap:", f"{check.mean():.6%}")


## Current risk: signed variance shares, three ETF legs

**Explained Risk (ER)** is a signed covariance share. At L3, market + sector + subsector +
residual ≈ 100% — and a single layer can print negative. That is information, not error:
see the callout below the chart.

**Hedge Ratio (HR)** is ETF dollars per $1 of stock, signed. **Hedge legs stop at
subsector** — market, sector, subsector are the tradeable layers; FF2 has no ETF hedge.


In [ ]:
dec, _ = api_post("/decompose", ticker=STOCK_TICKER)
exposure = dec.get("exposure") or {}
er = pd.Series({
    "Market": (exposure.get("market") or {}).get("er"),
    "Sector": (exposure.get("sector") or {}).get("er"),
    "Subsector": (exposure.get("subsector") or {}).get("er"),
    "Residual": (exposure.get("residual") or {}).get("er"),
}, dtype="float64").dropna()

hedge_rows = []
for layer_key, label in [("market", "Market"), ("sector", "Sector"), ("subsector", "Subsector")]:
    leg = exposure.get(layer_key) or {}
    hedge_rows.append({
        "Layer": label,
        "ETF": leg.get("hedge_etf"),
        "HR ($ ETF / $1 stock)": leg.get("hr"),
    })
hedge = pd.DataFrame(hedge_rows)
hedge["Notional hedge ($)"] = hedge["HR ($ ETF / $1 stock)"] * NOTIONAL_USD
hedge_display = hedge.copy()
hedge_display["HR ($ ETF / $1 stock)"] = hedge_display["HR ($ ETF / $1 stock)"].map(
    lambda x: f"{x:.3f}" if pd.notna(x) else "—")
hedge_display["Notional hedge ($)"] = hedge_display["Notional hedge ($)"].map(
    lambda x: f"${x:,.0f}" if pd.notna(x) else "—")
display(hedge_display)

fig, ax = plt.subplots(figsize=(9, 3.6))
labels = list(er.index)[::-1]
values = list(er.values)[::-1]
colors = [LAYER_COLORS.get(k, "#94A3B8") if v >= 0 else "#D97706" for k, v in zip(labels, values)]
ax.barh(labels, values, color=colors, height=0.58)
for k, v in zip(labels, values):
    ax.annotate(f"{v:+.1%}", (v, k), ha="left" if v >= 0 else "right",
                va="center", fontsize=9.5, fontweight="bold",
                color="#334155" if v >= 0 else "#B45309",
                xytext=(4 if v >= 0 else -4, 0), textcoords="offset points")
ax.axvline(0, color="#9CA3AF", lw=0.8)
ax.grid(axis="x", alpha=0.5)
ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Share of variance (signed covariance share)")
ax.set_title(f"{STOCK_TICKER} — current L3 Orth Risk decomposition (layers sum to {er.sum():.0%})", loc="left")
ax.margins(x=0.14)
plt.tight_layout()
plt.show()

if (er < 0).any():
    neg = ", ".join(er[er < 0].index)
    print(f"Negative layer ({neg}) — a signed covariance share, not an error. See the callout below.")


**Why can a variance share be negative?** ER is β·Cov(layer, stock) / Var(stock), not
an incremental R² — layers still sum to 100%, but any one can print negative. For a
mega-cap that dominates its own subsector ETF, the orthogonal subsector factor is
effectively "peers ex-this-name" — and the stock can trade **against** that spread. A
negative ER is a trade instruction, not a model artifact: shorting that leg "by the book"
would **add** variance, not remove it. The right response is a shallower hedge — which is
exactly what **L\*** automates, next.


## L\* — let the API pick the hedge depth

`GET /lstar` dispatches, per date, the **shallowest** cascade level whose marginal
explained-return clears a threshold (default 1%). Hedge legs stop at subsector — L\* only
dispatches among L1/L2/L3. The FF2 style block further splits the L\* residual into
*style* + *stock-specific* variance shares (a diagnostic on the L\* basis, no ETF hedge).

**For the full multi-year dispatch history and residual chart for one ticker, see
[`lstar_timeseries.ipynb`](./lstar_timeseries.ipynb)** — this cell shows today's snapshot:
the depth menu with dollar tickets side by side.


In [ ]:
lstar_body, _ = api_get("/lstar", ticker=STOCK_TICKER)
level_now = next((lvl for lvl in reversed(lstar_body["lstar"]) if lvl), None)
print(f"L* today for {STOCK_TICKER}: {level_now}  "
      f"(marginal-ER threshold {lstar_body.get('threshold_used', 0.01):.0%})")

hedge_levels = dec.get("hedge_levels") or {}
menu_rows = []
for lvl in ["L1", "L2", "L3"]:
    blk = hedge_levels.get(lvl) or {}
    etfs = blk.get("hedge_etfs") or {}
    legs = []
    for leg_name in ["market", "sector", "subsector"]:
        hr = blk.get(f"{leg_name}_hr")
        etf = etfs.get(leg_name)
        if hr is not None and etf:
            side = "short" if hr < 0 else "long"
            legs.append(f"{side} ${abs(hr) * NOTIONAL_USD:,.0f} {etf}")
    res_er = blk.get("residual_er")
    menu_rows.append({
        "Depth": lvl + ("  <- L*" if lvl == hedge_levels.get("recommended_level") else ""),
        "Residual ER": f"{res_er:.1%}" if res_er is not None else "—",
        f"Ticket on ${NOTIONAL_USD:,.0f}": " · ".join(legs) or "—",
    })
display(pd.DataFrame(menu_rows).set_index("Depth"))

style_ev = (dec.get("style") or {}).get("explained_variance")
ss_ev = (dec.get("stock_specific") or {}).get("explained_variance")
if style_ev is not None and ss_ev is not None:
    print(
        f"FF2 split of the L* residual: style {style_ev:.1%} + stock-specific {ss_ev:.1%} "
        f"≈ {style_ev + ss_ev:.1%} variance share (variance-ratio convention, L* basis). "
        "No ETF hedge on FF2 — the stock-specific share isolates stock-picking from style tilt."
    )


## Product surfaces: pre-rendered panels + PDF

These calls return **bytes from the API** — server-rendered panels and a tearsheet PDF,
the hosted counterparts of the charts above.


In [ ]:
PANEL_SLUGS = ("l3_explained_risk_hbar", "hedge_notionals_hbar", "hedge_depth_retained")
for slug in PANEL_SLUGS:
    try:
        payload, lineage = client.snapshot_panel("stock", STOCK_TICKER, slug, format="png")
        if not isinstance(payload, (bytes, bytearray)):
            print(f"{slug}: unexpected payload type {type(payload)}; skip display")
            continue
        out = OUTPUT_DIR / f"{STOCK_TICKER}_{slug}.png"
        out.write_bytes(payload)
        as_of = getattr(lineage, "data_as_of", None)
        note = f"  as_of={as_of}" if as_of else ""
        print(f"OK  {slug}  ({len(payload):,} bytes){note}  -> {out.name}")
        display(Image(data=bytes(payload)))
    except Exception as exc:
        print(f"SKIP {slug}: {exc}")

try:
    pdf_bytes, _ = client.get_metrics_snapshot_pdf(STOCK_TICKER)
    pdf_path = OUTPUT_DIR / f"{STOCK_TICKER}_metrics_snapshot.pdf"
    pdf_path.write_bytes(pdf_bytes)
    print(f"PDF {pdf_path.name} ({len(pdf_bytes):,} bytes)")
except Exception as exc:
    print(f"SKIP metrics snapshot PDF: {exc}")


## Part II · Fund — holdings-derived cascade through the FF2 layer

Fund search returns a stable `bw_fund_id`. The composed snapshot bundles registry
metadata, bitemporal dates, holdings-derived returns, diagnostics, history, and the hedge
basket. The fund return identity is the full sequence — market + sector + subsector +
**style (FF2)** + **final residual** (+ a small identity residual from aggregation).
Style is attributed, **not** credited as selection; it carries no ETF hedge.


In [ ]:
search_body, search_headers = api_get("/funds/search", q=FUND_QUERY, limit=10)
fund_hits = search_body.get("results") or []
if not fund_hits:
    raise ValueError(f"No fund matched {FUND_QUERY!r}.")

hits = pd.DataFrame(fund_hits)
show_cols = [c for c in ["ticker", "fund_name", "equity_style_9box", "morningstar_category",
                          "net_expense_ratio", "latest_report_date", "bw_fund_id"]
             if c in hits.columns]
display(hits[show_cols].head(10))

exact = hits[hits.get("ticker", pd.Series(index=hits.index, dtype=str)).astype(str).str.upper() == FUND_QUERY.upper()]
selected = (exact.iloc[0] if not exact.empty else hits.iloc[0]).to_dict()
FUND_ID = selected["bw_fund_id"]
FUND_LABEL = selected.get("ticker") or selected.get("fund_name") or FUND_ID
print("Selected:", FUND_LABEL, "|", selected.get("fund_name"), "|", FUND_ID)

fund_snapshot, fund_headers = api_get(f"/funds/snapshot/{FUND_ID}")
fund_metrics = fund_snapshot.get("metrics") or {}
fund_returns = fund_metrics.get("returns") or {}
diagnostics = fund_metrics.get("diagnostics") or {}

history_rows = ((fund_snapshot.get("portfolio_history") or {}).get("rows") or [])
style_val = fund_returns.get("style")
if style_val is None and history_rows:
    hist_tmp = pd.DataFrame(history_rows).dropna(subset=["portfolio_gross_return"])
    if not hist_tmp.empty and "portfolio_style_return" in hist_tmp.columns:
        style_val = hist_tmp.iloc[-1]["portfolio_style_return"]

steps = {
    "Market": fund_returns.get("market"),
    "Sector": fund_returns.get("sector"),
    "Subsector": fund_returns.get("subsector"),
    "Style (FF2)": style_val,
    "Final residual": fund_returns.get("idiosyncratic"),
}
steps = {k: v for k, v in steps.items() if v is not None}
gross = fund_returns.get("gross")

fig, ax = plt.subplots(figsize=(10, 4.4))
waterfall(ax, steps, total_label="Sum",
          title=f"{FUND_LABEL} — latest month, holdings-derived return decomposition")
plt.tight_layout()
plt.show()

if gross is not None:
    layer_sum = sum(steps.values())
    print(f"Gross {gross:.2%} | layer sum {layer_sum:.2%} | identity residual {gross - layer_sum:+.3%} "
          "(holdings-aggregation remainder)")


### Is the final residual — the stock-selection sleeve — persistent?

**Final residual** = the holdings-weighted selection sleeve after market / sector /
subsector **and** the FF2 style strip. One month is noise; the cumulative path over the
fund's holdings history is the more honest read — and it shows whether style and
selection are even pulling the same direction.


In [ ]:
fund_history = pd.DataFrame(history_rows)

if fund_history.empty:
    print("No portfolio history in this fund snapshot. Try another fund with a longer holdings panel.")
else:
    fund_history["teo"] = pd.to_datetime(fund_history["teo"])
    fund_history = fund_history.sort_values("teo").reset_index(drop=True)
    fund_history = fund_history.dropna(subset=["portfolio_gross_return"]).reset_index(drop=True)

    rename = {
        "portfolio_market_return": "Market",
        "portfolio_sector_return": "Sector",
        "portfolio_subsector_return": "Subsector",
        "portfolio_style_return": "Style (FF2)",
        "portfolio_idiosyncratic_return": "Final residual",
        "portfolio_gross_return": "Gross",
    }
    available = {k: v for k, v in rename.items() if k in fund_history.columns}
    fund_attr = fund_history[list(available)].rename(columns=available)

    cum_fund = cum_with_t0(fund_attr, fund_history["teo"])
    fig, ax = plt.subplots(figsize=(11.5, 5))
    attribution_lines(ax, cum_fund,
                      title=f"{FUND_LABEL} — cumulative fund attribution (arithmetic, anchored at 0)")
    plt.tight_layout()
    plt.show()

    if "Final residual" in fund_attr:
        resid = fund_attr["Final residual"].dropna()
        persistence = {
            "Observed months": float(len(resid)),
            "Positive final-residual months": (resid > 0).mean() if len(resid) else np.nan,
            "Cumulative final residual (selection)": resid.sum(),
        }
        if "Style (FF2)" in fund_attr:
            persistence["Cumulative style (FF2) contribution"] = fund_attr["Style (FF2)"].sum()
        display(pd.DataFrame({"Value": persistence}))


### Fund hedge basket

Same HR convention as stocks: ETF notional per $1 of fund exposure. The basket stops at subsector — style and final residual have no hedge legs by construction.


In [ ]:
hedge_block = fund_snapshot.get("hedge") or {}
hedge_rows = []
for level in ["L1", "L2", "L3"]:
    for leg in hedge_block.get(level) or []:
        hedge_rows.append({"Level": level, "ETF": leg.get("etf"), "HR ($ ETF / $1 fund)": leg.get("hr")})

fund_hedge = pd.DataFrame(hedge_rows)
if fund_hedge.empty:
    print("No fund hedge basket is available for this snapshot.")
else:
    fund_hedge["Notional hedge ($)"] = fund_hedge["HR ($ ETF / $1 fund)"] * NOTIONAL_USD
    plotted = fund_hedge.dropna(subset=["HR ($ ETF / $1 fund)"]).copy()
    top = plotted.reindex(plotted["HR ($ ETF / $1 fund)"].abs().sort_values(ascending=False).index).head(12)
    rest = plotted.drop(top.index)
    top = top.iloc[::-1]

    level_color = {"L1": LAYER_COLORS["Market"], "L2": LAYER_COLORS["Sector"], "L3": LAYER_COLORS["Subsector"]}
    fig, ax = plt.subplots(figsize=(10, 4.6))
    labels = (top["Level"] + " · " + top["ETF"].astype(str)).tolist()
    values = top["HR ($ ETF / $1 fund)"].tolist()
    colors = top["Level"].map(level_color).tolist()
    if not rest.empty:
        labels = [f"Other ({len(rest)} legs, net)"] + labels
        values = [rest["HR ($ ETF / $1 fund)"].sum()] + values
        colors = ["#CBD5E1"] + colors
    ax.barh(labels, values, color=colors, height=0.62)
    ax.axvline(0, color="#9CA3AF", lw=0.8)
    ax.grid(axis="x", alpha=0.5)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("ETF notional per $1 of fund exposure")
    ax.set_title(f"{FUND_LABEL} — largest hedge legs (top 12 of {len(plotted)} by |HR|)", loc="left")
    plt.tight_layout()
    plt.show()


## Part III · Bring your book

`analyze_portfolio` runs the same hierarchy on a weight vector — the pattern for hosting
Orth Risk next to an existing factor model. (`surface_stocks_sdk.ipynb` shows the bare
call; this cell adds the waterfall + PDF.)


In [ ]:
pa = client.analyze_portfolio(BOOK, metrics=["full_metrics", "hedge_ratios"], years=1)
print("Book:", BOOK)

er_port = getattr(pa, "portfolio_l3_er_weighted_mean", None)
hr_port = getattr(pa, "portfolio_hedge_ratios", None)
if er_port is not None:
    display(pd.DataFrame({"L3 ER (weight-mean)": er_port}).T if not isinstance(er_port, pd.DataFrame) else er_port)
if hr_port is not None:
    display(hr_port if isinstance(hr_port, pd.DataFrame) else pd.DataFrame({"portfolio_hedge_ratios": hr_port}))

if er_port is not None:
    if isinstance(er_port, pd.DataFrame):
        er_map = er_port.iloc[0].to_dict()
    elif isinstance(er_port, pd.Series):
        er_map = er_port.to_dict()
    else:
        er_map = dict(er_port)
    er_steps = {
        "Market": er_map.get("l3_market_er"),
        "Sector": er_map.get("l3_sector_er"),
        "Subsector": er_map.get("l3_subsector_er"),
        "Residual": er_map.get("l3_residual_er"),
    }
    er_steps = {k: float(v) for k, v in er_steps.items() if v is not None and pd.notna(v)}
    if er_steps:
        fig, ax = plt.subplots(figsize=(8.5, 3.8))
        waterfall(ax, er_steps, total_label="Total", title="Demo book — weight-mean L3 explained risk")
        plt.tight_layout()
        plt.show()

try:
    book_pdf, _ = client.post_portfolio_risk_snapshot_pdf(BOOK, title="Demo book")
    book_pdf_path = OUTPUT_DIR / "demo_book_risk_snapshot.pdf"
    book_pdf_path.write_bytes(book_pdf)
    print(f"Book PDF -> {book_pdf_path} ({len(book_pdf):,} bytes)")
except Exception as exc:
    print(f"SKIP book PDF: {exc}")


## Next steps

- **L\* over time, one ticker, deep dive:** [`lstar_timeseries.ipynb`](./lstar_timeseries.ipynb)
- **Full REST → SDK → AOM ladder:** [`riskmodels_quickstart.ipynb`](./riskmodels_quickstart.ipynb)
- **Funds & 13F, HTTP golden path:** [`surface_funds_http.ipynb`](./surface_funds_http.ipynb)
- **Stocks, SDK golden path:** [`surface_stocks_sdk.ipynb`](./surface_stocks_sdk.ipynb)
- **OpenAPI** (all paths / schemas): repo root `OPENAPI_SPEC.yaml`.

**Interpretation discipline:** the final residual is selection *after* the full sequence —
market, sector, subsector, and the FF2 style strip. A single window is noise; persistence
across a fund's holdings history is the claim worth testing.
